# Laboratory 6: Quantum Key Distribution — The BB84 Protocol

Two parties — **Alice** and **Bob** — want to agree on a secret key while an
eavesdropper — **Eve** — has full access to the channel connecting them. Classical
cryptography solves this with computational hardness assumptions. In 1984, Bennett and
Brassard showed that quantum mechanics offers something stronger: a protocol in which
**eavesdropping is physically detectable**, because measuring an unknown quantum state
disturbs it.

In this notebook you build BB84 from single-qubit Qiskit circuits: the four signal
states, basis-selective measurement, sifting, the quantum bit error rate (QBER), an
intercept-resend Eve, and the detection statistics that catch her.

### Learning objectives

1. **Explain the logic of BB84 and its security foundations** — random bases, mutually
   unbiased states, sifting.
2. **Perform and analyze QKD experiments** — measure the sifted fraction and QBER and
   compare them with the exact theoretical values.
3. **Understand the measurement–disturbance–security relationship** — derive and
   measure the 25% intercept-resend QBER and the detection probability
   $P_{\text{det}}(k) = 1 - (3/4)^k$.
4. **Discuss the relevance for cryptographic systems** — key rates, error thresholds,
   and what changes on real hardware.

**Prerequisites:** Lab 5 (Quantum Measurements) and the Quantum Communication course.
This notebook is the hands-on companion of the Lab 6 interactive guide — several
exercises mirror experiments you can replay in the guide's BB84 simulator (Batch mode).

**Reproducibility:** every random choice below is seeded (`SEED = 42`); running the
notebook top to bottom reproduces exactly the numbers shown.

> ### Convention note — bit ordering
>
> This lab series reports every quantum state in the **lab convention (big-endian)**:
> in $|q_0 q_1 \dots q_{n-1}\rangle$, qubit $0$ is the **leftmost** bit. **Qiskit uses
> the opposite (little-endian) order**, with qubit $0$ rightmost in returned
> bitstrings.
>
> Every register in this notebook is a **single qubit**, so the two conventions
> coincide: a 1-bit string reads the same in both, and the series' converter
> `qiskit_to_lab()` is the identity here. We define and use it anyway, and state this
> note, for consistency with the multi-qubit labs of the series — where the
> distinction genuinely bites (see the Lab 4 portfolio notebook).

### Environment Setup

In [ ]:
"""
Environment Setup Module.
Run this cell first to install all required libraries.
"""
%pip install -q qiskit qiskit-aer matplotlib

import numpy as np
import matplotlib.pyplot as plt

import qiskit
import qiskit_aer
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer import AerSimulator
from IPython.display import display

# All randomness in this notebook is seeded, so every run reproduces these numbers.
SEED = 42

BASIS_NAMES = np.array(["Z", "X"])   # basis flag 0 = Z, 1 = X


def qiskit_to_lab(bitstring: str) -> str:
    """Convert a Qiskit (little-endian) bitstring to lab (big-endian) convention.

    Identity on the 1-bit strings of this notebook — kept for series consistency.
    """
    return bitstring[::-1]


def lab_to_qiskit(bitstring: str) -> str:
    """Convert a lab (big-endian) bitstring to Qiskit (little-endian) convention."""
    return bitstring[::-1]


print("qiskit    ", qiskit.__version__)
print("qiskit-aer", qiskit_aer.__version__)
print("numpy     ", np.__version__)

---
## 1 · The Four BB84 States

**Why this step:** the entire protocol reduces to the measurement statistics of four
single-qubit states. Get these four states and their Born probabilities right, and
everything else — sifting, QBER, eavesdropper detection — follows by counting.

Alice encodes each bit $a \in \{0,1\}$ in a randomly chosen basis: the **Z-basis**
(computational) or the **X-basis** (superposition). Bit 0 always maps to the $+1$
eigenstate, bit 1 to the $-1$ eigenstate:

| | basis $Z$ | basis $X$ |
|---|---|---|
| bit 0 | $\|0\rangle$ | $\|{+}\rangle = (\|0\rangle + \|1\rangle)/\sqrt{2}$ |
| bit 1 | $\|1\rangle$ | $\|{-}\rangle = (\|0\rangle - \|1\rangle)/\sqrt{2}$ |

> **Lab 5 bridge.** These are exactly the states you placed on the Bloch sphere in
> Lab 5: the poles ($\theta = 0, \pi$) and two equator points
> ($\theta = \pi/2$, $\varphi = 0, \pi$). Their key property is **mutual
> unbiasedness**: writing $|z_0\rangle = |0\rangle$, $|z_1\rangle = |1\rangle$,
> $|x_0\rangle = |{+}\rangle$, $|x_1\rangle = |{-}\rangle$,
> $$|\langle z_j | x_l \rangle|^2 = \tfrac12 \quad \text{for all } j, l \in \{0,1\}.$$
> A measurement in the wrong basis is a fair coin flip: it extracts zero information
> about the encoded bit and destroys the state. That single fact is the security
> foundation of BB84.

In [ ]:
"""
BB84 state preparation (protocol signal states).

Encoding map: bit 0 -> +1 eigenstate, bit 1 -> -1 eigenstate of the chosen basis:
    |psi(0,Z)> = |0>,  |psi(1,Z)> = |1>,  |psi(0,X)> = |+>,  |psi(1,X)> = |->
"""

def prepare_bb84_state(bit: int, basis: str) -> QuantumCircuit:
    """Return a 1-qubit circuit preparing the BB84 state |psi(bit, basis)>."""
    if bit not in (0, 1):
        raise ValueError(f"bit must be 0 or 1, got {bit!r}")
    if basis not in ("Z", "X"):
        raise ValueError(f"basis must be 'Z' or 'X', got {basis!r}")
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)          # |0> -> |1>
    if basis == "X":
        qc.h(0)          # |0> -> |+>,  |1> -> |->
    return qc


# The four states, their circuits and statevectors
states = {}
for basis in ("Z", "X"):
    for bit in (0, 1):
        label = {("Z", 0): "|0>", ("Z", 1): "|1>",
                 ("X", 0): "|+>", ("X", 1): "|->"}[(basis, bit)]
        qc = prepare_bb84_state(bit, basis)
        sv = Statevector(qc)
        states[label] = np.asarray(sv)
        print(f"bit {bit}, basis {basis}  ->  {label:4s}  amplitudes: "
              f"[{sv[0]:.4f}, {sv[1]:.4f}]")
        assert abs(np.linalg.norm(sv) - 1.0) < 1e-12, "state must be normalized"

# Mutual unbiasedness check: |<z_j|x_l>|^2 = 1/2 for all four cross-overlaps
print("\nMutual unbiasedness |<z_j|x_l>|^2 (theory: all 0.5):")
for zj in ("|0>", "|1>"):
    row = []
    for xl in ("|+>", "|->"):
        overlap = abs(np.vdot(states[zj], states[xl])) ** 2
        assert abs(overlap - 0.5) < 1e-12, "bases must be mutually unbiased"
        row.append(f"{overlap:.12f}")
    print(f"  {zj}: {row}")
print("All checks passed: 4 normalized states, 4 cross-overlaps = 1/2 exactly.")

In [ ]:
"""
Bloch-sphere view of the four BB84 states (the Lab 5 picture).
Z-states at the poles, X-states on the equator - maximally 'sideways'
to each other, which is what mutual unbiasedness looks like geometrically.
"""
for label in ("|0>", "|1>", "|+>", "|->"):
    fig = plot_bloch_multivector(Statevector(states[label]), title=f"BB84 state {label}")
    display(fig)
    plt.close(fig)

---
## 2 · One Protocol Round: Measuring in a Chosen Basis

**Why this step:** Bob (and later Eve) must measure incoming qubits in a basis of
their choosing. Hardware gives us only Z-measurements — so the X-basis measurement
must be *built*, and how it is built matters.

Lab 5 distinguished two ways to "measure in the X-basis":

- **Procedure 1 — direct projection** onto $\{|{+}\rangle, |{-}\rangle\}$: the
  mathematical idealization. Post-measurement state: the X-eigenstate that was
  observed.
- **Procedure 2 — rotate, then Z-measure**: apply the canonical rotation
  $U_X = H$ (which maps $|{+}\rangle \mapsto |0\rangle$, $|{-}\rangle \mapsto
  |1\rangle$), then measure in Z. This is what real hardware executes.
  Post-measurement state: $|0\rangle$ or $|1\rangle$ — a **Z**-eigenstate.

The outcome *probabilities* are identical, so Bob — who keeps only the classical
bit — cannot tell the difference, and the Z-outcome $c$ maps directly to the key bit
$b = c$. The *post-measurement states* differ, which is invisible in this section but
becomes an observable attack signature in Section 4b. Everything below uses
**Procedure 2**, as hardware does ($U_Z = I$: a Z-measurement needs no rotation).

In [ ]:
"""
Basis-selective measurement, Procedure 2 (rotate-then-Z):
    Z-basis: measure directly              (U_Z = I)
    X-basis: apply U_X = H, then measure   (canonical rotation)
The Z-outcome c is the key bit: H maps |+> -> |0>, |-> -> |1>, so c = 0
means 'bit 0 eigenstate observed' in either basis.
"""

def measure_in_basis(qc: QuantumCircuit, basis: str) -> QuantumCircuit:
    """Append a basis-B measurement (Procedure 2) to a 1-qubit circuit, in place."""
    if basis == "X":
        qc.h(0)                  # canonical rotation U_X = H
    elif basis != "Z":
        raise ValueError(f"basis must be 'Z' or 'X', got {basis!r}")
    qc.measure_all()
    return qc


sim = AerSimulator()
SHOTS = 1000

# Matched basis: |+> measured in X -- deterministic (Born table: P(+)=1)
qc_match = measure_in_basis(prepare_bb84_state(0, "X"), "X")
counts_match = sim.run(qc_match, shots=SHOTS, seed_simulator=SEED).result().get_counts()
counts_match = {qiskit_to_lab(k): v for k, v in counts_match.items()}   # identity on 1 bit
print("matched   |+> measured in X:", counts_match, " (theory: all shots -> 0)")
assert counts_match.get("0", 0) == SHOTS, "matched-basis measurement must be deterministic"

# Mismatched basis: |+> measured in Z -- fair coin flip (P(0)=P(1)=1/2)
qc_mis = measure_in_basis(prepare_bb84_state(0, "X"), "Z")
counts_mis = sim.run(qc_mis, shots=SHOTS, seed_simulator=SEED + 1).result().get_counts()
counts_mis = {qiskit_to_lab(k): v for k, v in counts_mis.items()}
frac0 = counts_mis.get("0", 0) / SHOTS
print(f"mismatched |+> measured in Z: {counts_mis}  ->  P(0) = {frac0:.3f}"
      f"  (theory: 0.5, statistical tolerance +/-0.07 ~ 4.4 sigma)")
assert abs(frac0 - 0.5) < 0.07, "mismatched-basis outcome must be ~uniform"

display(plot_histogram([counts_match, counts_mis],
                       legend=["matched (X-meas of |+>)", "mismatched (Z-meas of |+>)"]))

---
## 3 · The Full Protocol: Sifting and the Error Rate

**Why this step:** one round is physics; the protocol is statistics. We now run $n$
rounds and measure the two numbers that define BB84's baseline behaviour.

Per round $i$: **(1)** Alice draws a uniform bit $a_i$ and basis $\alpha_i$, sends
$|\psi(a_i, \alpha_i)\rangle$; **(2)** Bob draws an independent uniform basis
$\beta_i$ and measures, getting $b_i$; **(3)** bases (never bits) are announced
publicly, and only rounds with $\alpha_i = \beta_i$ are kept — the **sifted key**;
**(4)** a random subset of sifted bits is compared publicly to estimate the error
rate (Section 5); **(5)** the rest is the raw key.

Two exact predictions on an ideal channel:

- **Sifted fraction:** $P(\alpha_i = \beta_i) = \tfrac14 + \tfrac14 = \tfrac12$ — half
  the rounds survive.
- **QBER without Eve:** matched-basis measurements are deterministic (Section 2), so
  the error rate of the sifted key is **exactly 0** — not merely small. On an ideal
  channel, *any* sifted-key error means something touched the qubits in transit.

In [ ]:
"""
Full BB84 protocol run - vectorized over rounds, seeded end to end.

Eve models (Section 4):
    eve=None                -- no eavesdropper
    eve="intercept-resend"  -- measures in a random basis gamma (Procedure 2),
                               re-encodes her post-state with U_gamma^dagger, resends;
                               equivalently: re-prepares |psi(e, gamma)>
    eve="naive"             -- measures the same way but FORGETS the re-encoding:
                               resends her literal post-measurement state |e>
Interception is per-round independent with probability p (default 1 = every round).

Draw order (fixed, for reproducibility): a, alpha, beta, [intercepted, gamma];
Aer seeds: seed for Eve's measurements, seed+1 for Bob's.
"""

def run_bb84(n_rounds: int, seed: int, eve: str | None = None, p: float = 1.0) -> dict:
    """Run n_rounds of BB84; return per-round arrays and summary statistics."""
    rng = np.random.default_rng(seed)
    a     = rng.integers(0, 2, n_rounds)     # Alice's bits
    alpha = rng.integers(0, 2, n_rounds)     # Alice's bases (0=Z, 1=X)
    beta  = rng.integers(0, 2, n_rounds)     # Bob's bases
    sim = AerSimulator()

    gamma = np.full(n_rounds, -1)
    e     = np.full(n_rounds, -1)
    intercepted = np.zeros(n_rounds, dtype=bool)

    if eve is not None:
        intercepted = rng.random(n_rounds) < p
        gamma = np.where(intercepted, rng.integers(0, 2, n_rounds), -1)
        idx = np.flatnonzero(intercepted)
        if len(idx):
            # Stage 1 -- Eve measures each intercepted qubit in her basis
            eve_circuits = [
                measure_in_basis(prepare_bb84_state(int(a[i]), BASIS_NAMES[alpha[i]]),
                                 BASIS_NAMES[gamma[i]])
                for i in idx
            ]
            res = sim.run(eve_circuits, shots=1, memory=True,
                          seed_simulator=seed).result()
            for j, i in enumerate(idx):
                e[i] = int(res.get_memory(j)[0])

    # Stage 2 -- Bob measures what arrives (original state, or Eve's resend)
    bob_circuits = []
    for i in range(n_rounds):
        if intercepted[i]:
            if eve == "naive":
                sent = prepare_bb84_state(int(e[i]), "Z")   # literal |e>, no re-encoding
            else:
                sent = prepare_bb84_state(int(e[i]), BASIS_NAMES[gamma[i]])
        else:
            sent = prepare_bb84_state(int(a[i]), BASIS_NAMES[alpha[i]])
        bob_circuits.append(measure_in_basis(sent, BASIS_NAMES[beta[i]]))
    res = sim.run(bob_circuits, shots=1, memory=True,
                  seed_simulator=seed + 1).result()
    b = np.array([int(res.get_memory(i)[0]) for i in range(n_rounds)])

    kept = alpha == beta
    m = int(kept.sum())
    errors = int(((b != a) & kept).sum())
    return dict(a=a, alpha=alpha, beta=beta, b=b, kept=kept,
                gamma=gamma, e=e, intercepted=intercepted,
                n=n_rounds, sifted=m, errors=errors,
                sifted_fraction=m / n_rounds,
                qber=errors / m if m else float("nan"))


N = 2000
run0 = run_bb84(N, SEED)

print("First 12 rounds (no Eve):")
print("  round  a  alpha  beta  kept  b")
for i in range(12):
    print(f"  {i:5d}  {run0['a'][i]}    {BASIS_NAMES[run0['alpha'][i]]}     "
          f"{BASIS_NAMES[run0['beta'][i]]}    {'yes' if run0['kept'][i] else ' - '}   "
          f"{run0['b'][i]}")

sigma_frac = (0.25 / N) ** 0.5
print(f"\nn = {N} rounds, no eavesdropper:")
print(f"  sifted fraction : {run0['sifted_fraction']:.4f}"
      f"   (theory 0.5, tolerance +/-{4*sigma_frac:.3f} ~ 4 sigma)")
print(f"  sifted key len  : {run0['sifted']}   (expected ~{N//2})")
print(f"  QBER            : {run0['qber']:.6f}   (theory: exactly 0)")
assert abs(run0["sifted_fraction"] - 0.5) < 4 * sigma_frac
assert run0["qber"] == 0.0, "no-Eve QBER must be EXACTLY zero on an ideal channel"
print("\nBoth predictions confirmed - and note the QBER is exactly 0.0, not just small.")

---
## 4 · Enter Eve: The Intercept-Resend Attack

**Why this step:** the security claim of BB84 is not "Eve learns nothing" — it is
"Eve cannot learn anything *without leaving statistical fingerprints*". The simplest
attack makes this quantitative.

Eve intercepts each qubit, measures it in a random basis $\gamma_i$, and must resend
*something* to Bob — she re-prepares the state her measurement produced
(**measure and re-prepare**, exactly what the two-stage circuits below do). Condition
on a sifted round and split on her guess:

- $\gamma = \alpha$ (prob. $\tfrac12$): she reads $e = a$ correctly and resends the
  identical state — **no error**.
- $\gamma \neq \alpha$ (prob. $\tfrac12$): her outcome is a coin flip and her resent
  state is a wrong-basis eigenstate; Bob's matched measurement of it is again a coin
  flip — **error with probability $\tfrac12$**.

$$Q = \tfrac12 \cdot 0 + \tfrac12 \cdot \tfrac12 = \tfrac14 = 25\%.$$

The same experiment runs in the lab guide's simulator (Batch mode, strategy
*intercept-resend*) — here you get the circuit-level version, plus the conditional
decomposition $[\,Q\,|\,\gamma{=}\alpha\,] = 0$ and
$[\,Q\,|\,\gamma{\neq}\alpha\,] = \tfrac12$ measured separately.

In [ ]:
"""
Intercept-resend attack: full interception (p = 1), random Eve basis.
Theory: QBER = 1/4 on the sifted key; conditionally 0 (basis guessed right)
and 1/2 (guessed wrong).
"""
run_ir = run_bb84(N, SEED + 2, eve="intercept-resend")

m = run_ir["sifted"]
sigma_q = (0.25 * 0.75 / m) ** 0.5
print(f"n = {N} rounds, intercept-resend Eve (p = 1):")
print(f"  sifted key len : {m}")
print(f"  QBER           : {run_ir['qber']:.4f}"
      f"   (theory 0.25, tolerance +/-{4*sigma_q:.3f} ~ 4 sigma)")
assert abs(run_ir["qber"] - 0.25) < 4 * sigma_q

# Conditional decomposition by Eve's basis guess (sifted rounds only)
kept = run_ir["kept"]
err = (run_ir["b"] != run_ir["a"]) & kept
guess_right = (run_ir["gamma"] == run_ir["alpha"]) & kept
guess_wrong = (run_ir["gamma"] != run_ir["alpha"]) & kept
q_right = err[guess_right].mean()
q_wrong = err[guess_wrong].mean()
print(f"  QBER | Eve guessed right basis : {q_right:.4f}   (theory: exactly 0)")
print(f"  QBER | Eve guessed wrong basis : {q_wrong:.4f}   (theory: 0.5, tol +/-0.09)")
assert q_right == 0.0, "right-basis interception must be error-free"
assert abs(q_wrong - 0.5) < 0.09
print("\nEve's dilemma, measured: whenever she guesses wrong, she coin-flips Bob's bit.")

---
## 4b · Optional: The Naive Resend — Post-Measurement States Matter

**Why this step:** in Section 2 the difference between Procedure 1 and Procedure 2
was invisible — same probabilities, and Bob keeps only a classical bit. Here it
becomes an *observable attack signature*.

Eve measures via Procedure 2 (as hardware forces her to): rotate with $U_\gamma$,
measure Z. Her post-measurement state is the **computational** state $|e\rangle$ —
not the $\gamma$-basis eigenstate. To run the correct attack of Section 4 she must
**re-encode**: apply $U_\gamma^\dagger$ (equivalently, re-prepare
$|\psi(e, \gamma)\rangle$) before resending.

Suppose she forgets, and naively resends $|e\rangle$ as-is:

- Alice's Z-rounds: $\gamma = Z$ resends correctly; $\gamma = X$ scrambles $e$ into a
  coin flip that Bob then reads deterministically — error $\tfrac12$. Contribution:
  $\tfrac12 \cdot \tfrac12 = \tfrac14$.
- Alice's X-rounds: Bob X-measures a Z-eigenstate — a coin flip *regardless of
  $\gamma$* — error $\tfrac12$. Contribution: $\tfrac12$.

$$Q_{\text{naive}} = \tfrac12\cdot\tfrac14 + \tfrac12\cdot\tfrac12 = \tfrac38 = 37.5\%
\neq 25\%.$$

Every outcome *probability* in Eve's apparatus is identical between the two variants —
only the post-measurement state differs, and it is exactly what she resends. As you
saw in the HTML simulator's Batch mode (strategy *naive resend*), the QBER converges
to 37.5% instead of 25%. Now confirm it with circuits.

In [ ]:
"""
Naive Procedure-2 resend: Eve forgets the U_gamma^dagger re-encoding and
resends her literal post-measurement state |e>. Theory: QBER = 3/8.
"""
run_nv = run_bb84(N, SEED + 3, eve="naive")

m_nv = run_nv["sifted"]
sigma_nv = (0.375 * 0.625 / m_nv) ** 0.5
print(f"n = {N} rounds, naive-resend Eve (p = 1):")
print(f"  QBER : {run_nv['qber']:.4f}   (theory 0.375, tolerance +/-{4*sigma_nv:.3f} ~ 4 sigma)")
assert abs(run_nv["qber"] - 0.375) < 4 * sigma_nv

print("\nSummary of the three channels measured so far:")
print(f"  no Eve             : QBER = {run0['qber']:.4f}   (theory 0)")
print(f"  intercept-resend   : QBER = {run_ir['qber']:.4f}   (theory 0.25)")
print(f"  naive resend       : QBER = {run_nv['qber']:.4f}   (theory 0.375)")
print("\nIdentical measurement probabilities, different post-states, different attack")
print("statistics - the Procedure 1 vs Procedure 2 distinction, made operational.")

---
## 5 · Catching Eve: Detection Probability

**Why this step:** a 25% error rate only matters if Alice and Bob *look*. The looking
is the public comparison of $k$ sacrificed check bits; this section measures how fast
that catches a full-interception Eve.

Each check bit disagrees independently with probability $Q = \tfrac14$, so Eve
survives all $k$ comparisons with probability $(3/4)^k$:

$$P_{\text{det}}(k) = 1 - \left(\tfrac34\right)^k .$$

**Trial design:** protocol rounds are i.i.d., so *disjoint consecutive blocks* of $k$
sifted bits are statistically independent detection experiments. One long protocol
run therefore gives us hundreds of trials at circuit-level fidelity — each block
plays the role of one key-exchange session's check subset.

In [ ]:
"""
Detection experiment: one long intercept-resend run, chopped into disjoint
blocks of k sifted bits; each block = one independent detection trial.
Theory: P_det(k) = 1 - (3/4)^k.
"""
K_CHECK = 10
run_det = run_bb84(6000, SEED + 4, eve="intercept-resend")

err_flags = ((run_det["b"] != run_det["a"])[run_det["kept"]])   # sifted-key error flags
n_blocks = len(err_flags) // K_CHECK
blocks = err_flags[: n_blocks * K_CHECK].reshape(n_blocks, K_CHECK)
detected = blocks.any(axis=1).mean()

theory = 1 - 0.75 ** K_CHECK
sigma_det = (theory * (1 - theory) / n_blocks) ** 0.5
print(f"sifted bits: {len(err_flags)}  ->  {n_blocks} disjoint trials of k = {K_CHECK}")
print(f"empirical P(detect) : {detected:.4f}")
print(f"theory 1-(3/4)^{K_CHECK}  : {theory:.4f}   (tolerance +/-{4*sigma_det:.3f} ~ 4 sigma)")
assert abs(detected - theory) < 4 * sigma_det

print("\nHow fast detection becomes certain:")
for k in (1, 5, 10, 20):
    print(f"  k = {k:2d} check bits ->  P_det = {1 - 0.75**k:8.4f}")

---
## 6 · Optional: Partial Interception — $Q(p) = p/4$

**Why this step:** Eve can trade information for stealth by intercepting only a
fraction $p$ of the rounds. Errors arise only on intercepted rounds, so by linearity
$Q(p) = p/4$ — she can *reduce* her fingerprint but never erase it while learning
anything. This is the same sweep the guide's simulator runs (Batch mode, *Sweep p*);
here it comes from circuits.

In [ ]:
"""
Interception-probability sweep: measured QBER vs the Q(p) = p/4 line.
"""
p_values = [0.0, 0.25, 0.5, 0.75, 1.0]
N_SWEEP = 1600
measured = []
for i, p in enumerate(p_values):
    r = run_bb84(N_SWEEP, SEED + 10 + i,
                 eve=None if p == 0 else "intercept-resend", p=p)
    measured.append(r["qber"])
    theory_q = p / 4
    tol = 0.07 if p > 0 else 0.0
    flag = "exact" if p == 0 else f"tol +/-{tol}"
    print(f"  p = {p:4.2f}:  QBER = {r['qber']:.4f}   (theory {theory_q:.4f}, {flag})")
    if p == 0:
        assert r["qber"] == 0.0
    else:
        assert abs(r["qber"] - theory_q) < tol

fig, ax = plt.subplots(figsize=(6.5, 4))
p_line = np.linspace(0, 1, 100)
ax.plot(p_line, p_line / 4, color="#f59e0b", lw=2, label="theory  Q(p) = p/4")
ax.scatter(p_values, measured, color="#1d4ed8", zorder=3, label="measured (Aer, seeded)")
ax.set_xlabel("interception probability p")
ax.set_ylabel("QBER")
ax.set_title("Partial interception: stealth never reaches zero fingerprint")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

---
## 7 · Optional: BB84 Statistics on Real Hardware

Everything above ran on a noiseless simulator, where "QBER > 0 ⟹ eavesdropper" is an
exact implication. Real devices break the ideal-channel assumption: gate errors,
readout errors and decoherence produce a **nonzero QBER with no Eve at all**.

Try it: run the Section 2 circuits (matched-basis rounds only) on an IBM Quantum
backend. The matched-basis error rate — exactly 0.0000 in Section 3 — becomes the
device's noise floor (typically a few percent on current hardware).

**Discussion — noise or Eve?** A single errored bit cannot tell you which. Deployed
QKD systems handle this the only way statistics allows: characterize the channel's
noise floor, then *abort if the measured QBER significantly exceeds it* — because an
intercept-resend Eve adds 25 percentage points, she can never hide under a
few-percent floor. The full security analysis (with error correction and privacy
amplification, beyond this lab) proves BB84 remains secure below a QBER threshold of
roughly 11%: below it, a secret key can still be distilled; above it, abort. The
threshold logic you practiced in Section 5 is exactly what real systems run.

The cell below is **off by default** (it needs an IBM Quantum account and queue
time). Set `RUN_ON_HARDWARE = True` and fill in your instance to run it.

In [ ]:
"""
Hardware run (optional, off by default): matched-basis BB84 rounds on an
IBM Quantum backend -> empirical hardware QBER with no eavesdropper.
Requires: pip install qiskit-ibm-runtime, and a saved IBM Quantum account.
"""
RUN_ON_HARDWARE = False

if not RUN_ON_HARDWARE:
    print("Skipped (RUN_ON_HARDWARE = False).")
    print("Enable it to measure the matched-basis error rate of a real device -")
    print("the noise floor that real QKD systems must distinguish from Eve.")
else:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    service = QiskitRuntimeService()             # uses your saved account
    backend = service.least_busy(operational=True, simulator=False)
    print("backend:", backend.name)

    # 40 matched-basis rounds: (bit, basis) drawn with the same seeded rng
    rng = np.random.default_rng(SEED)
    rounds = [(int(rng.integers(0, 2)), BASIS_NAMES[rng.integers(0, 2)])
              for _ in range(40)]
    circuits = [measure_in_basis(prepare_bb84_state(bit, basis), basis)
                for bit, basis in rounds]
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    jobs = SamplerV2(mode=backend).run(pm.run(circuits), shots=64)
    result = jobs.result()

    errors = total = 0
    for (bit, _basis), pub in zip(rounds, result):
        counts = pub.data.meas.get_counts()
        wrong = sum(v for key, v in counts.items() if int(qiskit_to_lab(key)) != bit)
        errors += wrong
        total += sum(counts.values())
    print(f"hardware matched-basis QBER: {errors/total:.4f}")
    print("(simulator value was exactly 0 - the difference is the device noise floor)")

---
## Summary

| Quantity | Theory | Measured in | Also in the guide's simulator |
|---|---|---|---|
| Sifted fraction | $1/2$ | Section 3 | Batch mode, any run |
| QBER, no Eve (ideal channel) | $0$ exactly | Section 3 | Batch mode, Eve off |
| QBER, intercept-resend | $1/4$ | Section 4 | Batch mode, *intercept-resend* |
| QBER split by Eve's guess | $0$ and $1/2$ | Section 4 | Step-by-step log |
| QBER, naive resend | $3/8$ | Section 4b | Batch mode, *naive resend* |
| Detection probability | $1-(3/4)^k$ | Section 5 | Detection panel |
| Partial interception | $Q(p) = p/4$ | Section 6 | *Sweep p* |

**Takeaways.** The sifted key costs half the rounds; on an ideal channel it is
error-free, so errors *are* the eavesdropper signal. Intercept-resend buys Eve
partial information at an unavoidable 25% QBER, detection of which becomes
exponentially certain at $1-(3/4)^k$; even partial interception scales the
fingerprint only linearly, never to zero. And the quiet star of the lab: the
Procedure 1 / Procedure 2 distinction from Lab 5 — invisible to Bob — decides whether
Eve's attack produces 25% or 37.5% errors, because what she resends is a
post-measurement state. Measurement disturbance is not a nuisance here; it *is* the
security.

After the error check, the usable key is what remains of the sifted key: $m - k$
bits, minus the further costs of error correction and privacy amplification on real
channels — the subject of the security-proof literature (QBER threshold ≈ 11%).